# 06 Standardization

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats


from sklearn.preprocessing import StandardScaler

In [ ]:
data = gpd.read_parquet(
    "/data/uscuni-restricted/geodem/05_data/_merged_census_2021.parquet"
)
total = pd.read_csv(
    "/data/uscuni-restricted/geodem/04_spatial/total.csv",
    dtype={"nadzsjd": str},
    index_col=0,
)
hh = pd.read_csv(
    "/data/uscuni-restricted/geodem/04_spatial/hh_total.csv",
    dtype={"nadzsjd": str},
    index_col=0,
)

In [ ]:
cols_no_division = [
    "Počet obyvatel na obytnou plochu",
    "Počet obyvatel na dům",
    "Počet obyvatel na byt",
]

In [ ]:
data_total = data.join(total)
data_hh_total = data_total.join(hh)

In [ ]:
data_relative = data_hh_total.drop(data.columns[:12], axis=1)
cols_numeric = data_relative.columns.drop("geometry")
data_relative[cols_numeric] = data_relative[cols_numeric].astype(float)

In [ ]:
pop_total = data_relative["Obyvatelstvo celkem"].replace(0, np.nan)
hh_total = data_relative["Hospodařící domácnosti v bytech celkem"].replace(0, np.nan)

In [ ]:
cols_domacnosti = [col for col in cols_numeric if "HD" in col]

In [ ]:
exclude_from_norm = cols_no_division + [
    "Obyvatelstvo celkem",
    "Hospodařící domácnosti v bytech celkem",
]

In [ ]:
cols_to_norm_pop = [
    col
    for col in cols_numeric
    if col not in cols_domacnosti and col not in exclude_from_norm
]

In [ ]:
data_relative[cols_domacnosti] = (
    data_relative[cols_domacnosti].div(hh_total, axis=0) * 100
)
data_relative[cols_to_norm_pop] = (
    data_relative[cols_to_norm_pop].div(pop_total, axis=0) * 100
)

In [ ]:
all_target_cols = cols_domacnosti + cols_to_norm_pop

In [ ]:
data_relative.replace([np.inf, -np.inf], np.nan, inplace=True)
data_relative.dropna(subset=all_target_cols + cols_no_division, inplace=True)
data_relative[all_target_cols + cols_no_division] = data_relative[
    all_target_cols + cols_no_division
].clip(-1e6, 1e6)

In [ ]:
plt.figure(figsize=(20, 10))


sns.boxplot(data=data_relative[all_target_cols])
plt.xticks(rotation=90, ha="right")

In [ ]:
z_scores = np.abs(stats.zscore(data_relative[all_target_cols]))
# Find rows where any column has a Z-score > 3
outliers = data_relative[(z_scores > 3).any(axis=1)]
print(f"Number of outliers detected: {len(outliers)}")

In [ ]:
data_relative[all_target_cols] = StandardScaler().fit_transform(
    data_relative[all_target_cols]
)
data_relative[cols_no_division] = StandardScaler().fit_transform(
    data_relative[cols_no_division]
)

In [ ]:
admin = data.iloc[:, :12]

In [ ]:
dataset = admin.join(data_relative)

In [ ]:
dataset = dataset.dropna(subset=["geometry"])

In [ ]:
dataset = gpd.GeoDataFrame(dataset, geometry="geometry")

In [ ]:
dataset.to_parquet("dat_f_m.parquet")